# Bài tập Gradient Descent & Perceptron (3.26 – 3.30)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

## Bài 3.26 – Gradient Descent
Hàm số: $f(x) = x^2 - 4x + 5$, $x_0 = 5$, $\eta = 0.2$.

**1. Đạo hàm:** $f'(x) = 2x - 4$

Quy tắc cập nhật: $x_{t+1} = x_t - \eta \cdot f'(x_t)$

In [ ]:
# Hàm số và đạo hàm
def f(x):
    return x**2 - 4*x + 5

def grad(x):
    return 2*x - 4

x0, eta, n_steps = 5, 0.2, 4

# 2 & 3. Thực hiện 4 bước cập nhật, lưu lại x_t, f'(x_t), f(x_t)
xs = [x0]
for t in range(n_steps):
    xs.append(xs[-1] - eta * grad(xs[-1]))

bang = pd.DataFrame({
    "t": range(n_steps + 1),
    "x_t": xs,
    "f'(x_t)": [grad(x) for x in xs],
    "f(x_t)": [f(x) for x in xs],
})
print(bang.to_string(index=False))

In [ ]:
# 4. Vẽ đồ thị f(x) và các điểm x_t
xx = np.linspace(-1, 6, 200)
plt.figure(figsize=(7, 4))
plt.plot(xx, f(xx), label="f(x) = x² − 4x + 5")
plt.plot(xs, [f(x) for x in xs], "ro--", label="Các bước GD")
for t, x in enumerate(xs):
    plt.annotate(f"x{t}", (x, f(x)), textcoords="offset points", xytext=(5, 8))
plt.scatter([2], [f(2)], c="g", marker="*", s=200, label="Cực tiểu x* = 2", zorder=5)
plt.xlabel("x"); plt.ylabel("f(x)"); plt.legend(); plt.grid(alpha=.3)
plt.title("Gradient Descent với η = 0.2")
plt.show()

In [ ]:
# 5. So sánh với nghiệm chính xác: f'(x) = 0  <=>  2x - 4 = 0  <=>  x* = 2
x_star = 2
print(f"Nghiệm chính xác: x* = {x_star}, f(x*) = {f(x_star)}")
print(f"Sau 4 bước:       x4 = {xs[-1]:.4f}, f(x4) = {f(xs[-1]):.4f}")
print(f"Sai số |x4 - x*| = {abs(xs[-1] - x_star):.4f}")
# Khoảng cách tới x* giảm theo hệ số (1 - 2η) = 0.6 sau mỗi bước
print("Tỉ số |x_{t+1}-x*| / |x_t-x*|:", [round((xs[i+1]-2)/(xs[i]-2), 2) for i in range(n_steps)])

### Nhận xét/Kết luận
- $x_t$ giảm dần: 5 → 3.8 → 3.08 → 2.648 → 2.3888, tiến về cực tiểu $x^* = 2$. $f(x_t)$ cũng giảm dần về $f(x^*) = 1$.
- Ta có $x_{t+1} - 2 = (1 - 2\eta)(x_t - 2) = 0.6\,(x_t - 2)$. Vì $|1 - 2\eta| = 0.6 < 1$ nên thuật toán **hội tụ** (tuyến tính, sai số giảm 40% mỗi bước) và không dao động.
- Điều kiện hội tụ với hàm này là $0 < \eta < 1$. Nếu $\eta \ge 1$ thì GD dao động hoặc phân kỳ. Với 4 bước, $x_4$ vẫn còn cách $x^*$ khoảng 0.39, cần thêm vài bước nữa để sát nghiệm.

## Bài 3.27 – Perceptron (dự đoán)
$w = [1, 2, -10]^T$, $x = [3, 4, 1]^T$ (đã thêm bias), $y = -1$.

In [ ]:
w = np.array([1, 2, -10])
x = np.array([3, 4, 1])
y = -1

# 1. Tính w^T x
wx = w @ x
print("1. w^T x =", " + ".join(f"({a})·({b})" for a, b in zip(w, x)), "=", wx)

# 2. Nhãn dự đoán: 1 nếu w^T x >= 0, ngược lại -1
y_hat = 1 if wx >= 0 else -1
print("2. Nhãn dự đoán ŷ =", y_hat)

# 3. Kiểm tra phân lớp sai: y · w^T x <= 0
print(f"3. y · w^T x = {y} · {wx} = {y*wx}")
print("   => Phân lớp SAI" if y*wx <= 0 else "   => Phân lớp ĐÚNG")

### Nhận xét/Kết luận
$w^Tx = 3 + 8 - 10 = 1 \ge 0$ nên mô hình dự đoán $\hat y = 1$. Nhãn thực tế là $y = -1$, và $y \cdot w^Tx = -1 \le 0$, vậy điểm dữ liệu **bị phân lớp sai**. Điểm này nằm sát biên quyết định ($w^Tx = 1$), chỉ cần một bước cập nhật là có thể sửa được.

## Bài 3.28 – Perceptron (cập nhật)
$w = [-2, 1, 0]^T$, $x = [2, 3, 1]^T$ (đã thêm bias), $y = 1$, $\eta = 1$.

In [ ]:
w = np.array([-2, 1, 0])
x = np.array([2, 3, 1])
y, eta = 1, 1

# 1. Kiểm tra phân lớp sai
wx = w @ x
print(f"1. w^T x = {wx},  y · w^T x = {y*wx}")
sai = y * wx <= 0
print("   => Phân lớp SAI" if sai else "   => Phân lớp ĐÚNG")

# 2. Nếu sai: cập nhật w <- w + η·y·x
if sai:
    w_new = w + eta * y * x
    print(f"2. w_mới = {w} + {eta}·{y}·{x} = {w_new}")
else:
    w_new = w

# 3. Tính lại w^T x sau cập nhật
wx_new = w_new @ x
print(f"3. w_mới^T x = {wx_new},  y · w_mới^T x = {y*wx_new}")
print("   => Sau cập nhật: phân lớp ĐÚNG" if y*wx_new > 0 else "   => Vẫn phân lớp SAI")

### Nhận xét/Kết luận
Ban đầu $w^Tx = -4 + 3 + 0 = -1$, nên $y \cdot w^Tx = -1 \le 0$ và mẫu **bị phân lớp sai**. Sau một bước cập nhật, $w_{mới} = [0, 4, 1]^T$ và $w_{mới}^Tx = 13 > 0$, mẫu đã được **phân lớp đúng**. Phép cập nhật cộng thêm $\|x\|^2 = 14$ vào $y \cdot w^Tx$, đẩy biên quyết định về phía làm đúng mẫu này.

## Bài 3.29 – Cài đặt lớp Perceptron
Cài đặt từ đầu bằng numpy. Trong mỗi epoch, thuật toán duyệt dữ liệu theo thứ tự ngẫu nhiên và cập nhật $w \leftarrow w + \eta\, y_i x_i$ khi gặp điểm sai ($y_i w^Tx_i \le 0$).

In [ ]:
class Perceptron:
    def __init__(self, learning_rate=1.0, max_iter=1000, random_state=None):
        self.learning_rate = learning_rate
        self.max_iter = max_iter          # số epoch tối đa
        self.random_state = random_state

    @staticmethod
    def _add_bias(X):
        # Thêm cột 1 vào cuối để gộp bias vào w
        return np.hstack([X, np.ones((X.shape[0], 1))])

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        # Nếu nhãn là {0, 1} thì chuyển sang {-1, 1}
        self.classes_ = np.unique(y)
        if set(self.classes_) <= {0, 1}:
            y = np.where(y == 1, 1, -1)
        Xb = self._add_bias(X)
        rng = np.random.default_rng(self.random_state)
        self.w_ = rng.normal(0, 0.01, Xb.shape[1])   # khởi tạo w nhỏ ngẫu nhiên
        self.errors_ = []                            # số lỗi ở mỗi epoch

        for epoch in range(self.max_iter):
            n_err = 0
            for i in rng.permutation(len(y)):
                if y[i] * (Xb[i] @ self.w_) <= 0:     # điểm bị phân lớp sai
                    self.w_ += self.learning_rate * y[i] * Xb[i]
                    n_err += 1
            self.errors_.append(n_err)
            if n_err == 0:                           # không còn điểm sai -> dừng
                break
        self.n_iter_ = len(self.errors_)
        return self

    def decision_function(self, X):
        return self._add_bias(np.asarray(X, dtype=float)) @ self.w_

    def predict(self, X):
        pred = np.where(self.decision_function(X) >= 0, 1, -1)
        # Trả nhãn về dạng {0, 1} nếu dữ liệu huấn luyện dùng {0, 1}
        if set(self.classes_) <= {0, 1}:
            pred = np.where(pred == 1, 1, 0)
        return pred

In [ ]:
# Kiểm thử trên dữ liệu 2D tách được tuyến tính
rng = np.random.default_rng(42)
N = 50
X_pos = rng.normal([2, 2], 0.8, (N, 2))
X_neg = rng.normal([-2, -2], 0.8, (N, 2))
X2d = np.vstack([X_pos, X_neg])
y2d = np.array([1]*N + [-1]*N)

model = Perceptron(learning_rate=1.0, max_iter=100, random_state=0).fit(X2d, y2d)
print("w (w1, w2, bias) =", model.w_)
print("Số epoch:", model.n_iter_, "| Số lỗi mỗi epoch:", model.errors_)
print("Accuracy trên tập huấn luyện:", (model.predict(X2d) == y2d).mean())

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(*X_pos.T, c="b", label="y = 1")
ax[0].scatter(*X_neg.T, c="r", marker="x", label="y = -1")
w1, w2, b = model.w_
xl = np.linspace(X2d[:, 0].min(), X2d[:, 0].max(), 100)
ax[0].plot(xl, -(w1*xl + b)/w2, "k-", label="Đường phân chia")
ax[0].set_title("Dữ liệu và đường phân chia"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(range(1, model.n_iter_+1), model.errors_, "o-")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Số lỗi"); ax[1].set_title("Số lỗi theo epoch"); ax[1].grid(alpha=.3)
plt.show()

### Nhận xét/Kết luận
Với dữ liệu tách được tuyến tính, Perceptron hội tụ sau vài epoch: số lỗi giảm về 0 và đường phân chia tách đúng hoàn toàn hai lớp. Điều này khớp với định lý hội tụ của Perceptron. Nếu dữ liệu không tách được tuyến tính, thuật toán sẽ không bao giờ đạt 0 lỗi và chỉ dừng khi hết `max_iter`.

## Bài 3.30 – Phân lớp nhị phân: Breast Cancer
Bài toán dùng tập dữ liệu `load_breast_cancer` của sklearn (569 mẫu, 30 đặc trưng). Trong sklearn gốc, nhãn 0 là ác tính và 1 là lành tính. Ở đây ta đổi lại để **lớp dương (1) là khối u ác tính**, vì đó là lớp cần phát hiện, và Precision/Recall sẽ tính theo lớp này.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import Perceptron as SkPerceptron

data = load_breast_cancer()
X = data.data
y = np.where(data.target == 0, 1, -1)   # 1 = ác tính, -1 = lành tính

# Mô tả dữ liệu
print("Kích thước:", X.shape)
print("Số mẫu ác tính (1):", (y == 1).sum(), "| lành tính (-1):", (y == -1).sum())
pd.DataFrame(X, columns=data.feature_names).describe().T.head(10)

In [ ]:
# Chia train/test 80/20 có stratify và chuẩn hóa (fit scaler chỉ trên tập train)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)
print("Train:", X_train_s.shape, "| Test:", X_test_s.shape)

In [ ]:
# Tìm cấu hình tốt nhất bằng 5-fold cross-validation trên tập train, chọn theo F1-score
learning_rates = [0.001, 0.01, 0.1, 1.0]
max_iters = [5, 10, 20, 50, 100]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for lr in learning_rates:
    for mi in max_iters:
        f1s = []
        for tr, va in skf.split(X_train, y_train):
            # Chuẩn hóa trong từng fold để tránh rò rỉ dữ liệu
            sc = StandardScaler().fit(X_train[tr])
            m = Perceptron(learning_rate=lr, max_iter=mi, random_state=0).fit(sc.transform(X_train[tr]), y_train[tr])
            f1s.append(f1_score(y_train[va], m.predict(sc.transform(X_train[va]))))
        results.append({"learning_rate": lr, "max_iter": mi, "F1_CV_mean": np.mean(f1s), "F1_CV_std": np.std(f1s)})

cv_df = pd.DataFrame(results).sort_values("F1_CV_mean", ascending=False)
print(cv_df.head(10).to_string(index=False))
best = cv_df.iloc[0]
print(f"\n=> Cấu hình tốt nhất: learning_rate = {best.learning_rate}, max_iter = {int(best.max_iter)}")

In [ ]:
# Huấn luyện mô hình tốt nhất trên toàn bộ tập train, đánh giá trên tập test
def danh_gia(ten, y_true, y_pred):
    return {"Mô hình": ten,
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred),
            "Recall": recall_score(y_true, y_pred),
            "F1-score": f1_score(y_true, y_pred)}

my_model = Perceptron(learning_rate=best.learning_rate, max_iter=int(best.max_iter), random_state=0).fit(X_train_s, y_train)
y_pred_my = my_model.predict(X_test_s)

# So sánh với sklearn.linear_model.Perceptron
sk_model = SkPerceptron(max_iter=1000, random_state=0).fit(X_train_s, y_train)
y_pred_sk = sk_model.predict(X_test_s)

bang_kq = pd.DataFrame([danh_gia("Perceptron tự cài (tốt nhất)", y_test, y_pred_my),
                        danh_gia("sklearn Perceptron", y_test, y_pred_sk)])
print(bang_kq.round(4).to_string(index=False))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, yp, ten in zip(ax, [y_pred_my, y_pred_sk], ["Perceptron tự cài", "sklearn Perceptron"]):
    cm = confusion_matrix(y_test, yp, labels=[1, -1])
    ConfusionMatrixDisplay(cm, display_labels=["Ác tính (1)", "Lành tính (-1)"]).plot(ax=a, colorbar=False)
    a.set_title(ten)
plt.tight_layout(); plt.show()

### Nhận xét/Kết luận
- Sau khi chuẩn hóa, cả hai mô hình đều đạt kết quả tốt (Accuracy > 92%), vì dữ liệu gần như tách được tuyến tính.
- Cấu hình tốt nhất theo CV là `learning_rate = 0.01`, `max_iter = 5` (F1-CV ≈ 0.96). Số epoch ít cho kết quả tốt hơn vì dữ liệu không tách được hoàn toàn: chạy lâu hơn thì w bị cập nhật qua lại bởi các điểm nhiễu. Dừng sớm lúc này có tác dụng như điều chuẩn.
- Trên tập test, **sklearn Perceptron tốt hơn** (F1 ≈ 0.95 so với 0.90 của mô hình tự cài). Nguyên nhân là sklearn có tiêu chí dừng theo độ giảm lỗi (`tol`), nên nghiệm ổn định hơn. Mô hình tự cài nhạy với thứ tự duyệt dữ liệu và với epoch cuối cùng. Chênh lệch trên tập test nhỏ (114 mẫu) chỉ tương ứng vài mẫu dự đoán sai.
- Ý nghĩa các độ đo trong bài toán này:
  - **Accuracy:** tỉ lệ dự đoán đúng trên toàn bộ mẫu.
  - **Precision:** trong số ca mô hình báo là "ác tính", có bao nhiêu ca thật sự ác tính. Precision thấp nghĩa là báo động nhầm nhiều.
  - **Recall:** trong số ca ác tính thật, mô hình phát hiện được bao nhiêu. Đây là độ đo **quan trọng nhất** trong y tế, vì bỏ sót ca ác tính gây hậu quả nghiêm trọng.
  - **F1-score:** trung bình điều hòa của Precision và Recall, cân bằng giữa hai độ đo. Đây là tiêu chí dùng để chọn mô hình.
- Perceptron không có cơ chế tối ưu biên (margin) và kết quả phụ thuộc vào thứ tự duyệt dữ liệu. Để cải thiện thêm, có thể dùng Pocket Perceptron, Logistic Regression hoặc SVM.